## Импорт библиотек ##

In [3]:
from pypdf import PdfReader
import re
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

## Загрузка данных ##

In [4]:
reader = PdfReader('Report.pdf')

In [5]:
text = ''

In [6]:
for page in reader.pages:
    text += page.extract_text() + "\n"


In [7]:
text

'Выводы по кластеризации \nПри использовании алгоритма K -средних оптимальное количество кластеров было \nопределено с помощью метода локтя и метода силуэта. Оба подхода показали \nцелесообразность выделения 4 кластеров.  \nПри использовании DBSCAN количество кластеров заранее не задавалось. Оно \nопределялось на основе двух параметров алгоритма: минимального количества соседей и \nрасстояния, в пределах которого объекты считаются принадлежащими к одной плотной \nобласти. В результате DBSCAN выделил 2 основные плотны е группы, одна из которых, в \nсвою очередь, может быть разделена на 3 подгруппы — одну крупную и две меньшего \nразмера. \nТаким образом, результаты двух методов дополняют друг друга. DBSCAN позволяет \nопределить общую структуру и плотность данных, выделяя основ ные плотные группы и \nих внутреннюю структуру. K -средних используется для более детальной сегментации и \nпозволяет выделить 4 устойчивые группы пользователей.  \nВ дальнейшем для основной сегментации используе

## Очистка текста ##

def clean_text(text):
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"\n(?=[а-яёa-z0-9-])", " ", text)
    text = re.sub(r"\s+([,.;:!?-])", r"\1", text)
    text = re.sub(r"\s+-\s*", "-", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = "\n".join(line.strip() for line in text.splitlines())

    
    fixes = {
        "плотны е": "плотные",
        "основ ные": "основные",
        "Ск орее": "Скорее",
        "пользовате лей": "пользователей",
        "пользова телей": "пользователей",
    }
    for bad, good in fixes.items():
        text = text.replace(bad, good)

   
    text = re.sub(
        r"(Кластер \d+ — [^\n.]+?) (Это|Эти|По|Основная|Для|Пользователи|Средний)",
        r"\1\n\2",
        text
    )

    
    text = re.sub(r"\n*(?=Кластер \d+ —)", "\n\n", text)

   
    text = re.sub(r"^(Выводы[^\n]+)\n(?=[А-ЯЁ])", r"\1\n\n", text)
    text = re.sub(r"(K-means[^\n]*:)\n(?!\n)", r"\1\n\n", text)

   
    text = re.sub(
        r"\.\n(Для этого сегмента|Основная цель|Также стоит|Полезными)",
        r".\n\n\1",
        text
    )

   
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.lstrip("\n")

    return text.strip()

In [36]:
def clean_text(text):
    
    text = text.replace('\n', ' ')
    
    
    text = re.sub(r'\s+', ' ', text)
    
   
    text = re.sub(r'\s*-\s*', '-', text)
    
    
    text = re.sub(r'\s+([,.!?;:])', r'\1', text)

    text = re.sub(r'(\w)[\r\n]+(\w)', r'\1\2', text)
    
    return text.strip()
    

In [37]:
text =  clean_text(text)

In [38]:
text

'Выводы по кластеризации При использовании алгоритма K-средних оптимальное количество кластеров было определено с помощью метода локтя и метода силуэта. Оба подхода показали целесообразность выделения 4 кластеров. При использовании DBSCAN количество кластеров заранее не задавалось. Оно определялось на основе двух параметров алгоритма: минимального количества соседей и расстояния, в пределах которого объекты считаются принадлежащими к одной плотной области. В результате DBSCAN выделил 2 основные плотны е группы, одна из которых, в свою очередь, может быть разделена на 3 подгруппы — одну крупную и две меньшего размера. Таким образом, результаты двух методов дополняют друг друга. DBSCAN позволяет определить общую структуру и плотность данных, выделяя основ ные плотные группы и их внутреннюю структуру. K-средних используется для более детальной сегментации и позволяет выделить 4 устойчивые группы пользователей. В дальнейшем для основной сегментации используем K-средних, а результаты DBSCAN

## Разделение на чанки ##

In [39]:
parts = re.split(r'(?=Кластер \d+ —)', text)

In [40]:
chunks = [p.strip() for p in parts if p.strip()]

In [50]:
chunks

['Выводы по кластеризации При использовании алгоритма K-средних оптимальное количество кластеров было определено с помощью метода локтя и метода силуэта. Оба подхода показали целесообразность выделения 4 кластеров. При использовании DBSCAN количество кластеров заранее не задавалось. Оно определялось на основе двух параметров алгоритма: минимального количества соседей и расстояния, в пределах которого объекты считаются принадлежащими к одной плотной области. В результате DBSCAN выделил 2 основные плотны е группы, одна из которых, в свою очередь, может быть разделена на 3 подгруппы — одну крупную и две меньшего размера. Таким образом, результаты двух методов дополняют друг друга. DBSCAN позволяет определить общую структуру и плотность данных, выделяя основ ные плотные группы и их внутреннюю структуру. K-средних используется для более детальной сегментации и позволяет выделить 4 устойчивые группы пользователей. В дальнейшем для основной сегментации используем K-средних, а результаты DBSCA

## Векторное преобразование ##

In [13]:
model = SentenceTransformer("intfloat/multilingual-e5-small")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [41]:
passages = [f'passage: {chunk}' for chunk in chunks]

In [42]:
embeddings = model.encode(passages, normalize_embeddings = True).astype('float32')

In [43]:
index = faiss.IndexFlatIP(embeddings.shape[1])

In [44]:
index.add(embeddings)

In [51]:
user_query = "работающие с вебинарами"

In [52]:
query_emb = model.encode([f'query: {user_query}'], normalize_embeddings =True).astype('float32')

In [53]:
D,I = index.search(query_emb,k=3)

In [55]:
D, I = index.search(query_emb, k=1)

for score, i in zip(D[0], I[0]):
    if i == -1:
        continue
    
    print("score:", score)
    print(chunks[i])
    print("-----")

score: 0.8677475
Кластер 1 — пользователи, работающие с вебинарами Эти пользователи заходят на платформу реже, чем кластер 0: средний промежуток между входами составляет около 20,6 часа. Основная часть их активности приходится на вебинары, тогда как домашними заданиями и оценками они занимаются значительно меньше. В основном они проводят и запускают вебинары. Для этого сегмента стоит сделать основной акцент на инструментах для работы с вебинарами. На главном экране можно показывать ближайшие вебинары, расписание и быстрый запуск или создание вебинара. Полезными будут напоминания о предстоящих вебинарах, изменениях в расписании и регистрации участников. Также можно упростить создание вебинаров с помощью шаблонов и добавить удобную работу с записями и материалами после проведения. Основная цель — сделать проведение вебинаров максимально простым и удобным.
-----
